[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/evaluation.ipynb)

# 08 . Evaluating LLMs (perplexity, benchmarks, harness, bootstrap CIs)

## Concept — Evaluating Language Models (revision notes)

**What it is**
- Measuring how good a language model is, with numbers you can trust: **perplexity** (how well it predicts held-out text), **benchmark metrics** (accuracy on multiple choice, exact match on generation), an **eval harness** that runs many tasks uniformly, and **confidence intervals** that tell you whether a difference between two models is real or noise.

**Why it matters**
- Fine-tuning, quantization or any change is only meaningful if you can measure its effect. Most evaluation mistakes are statistical (tiny test sets, comparing numbers that differ by noise) or procedural (padding in the average, train/test leakage, inconsistent normalisation), not mathematical.

**When to use**
- Perplexity: fast, task-agnostic regression test when the tokenizer and data are fixed. Task metrics: when you care about specific capabilities. Bootstrap CIs: always, when reporting results on a finite test set.

**Math & intuition**
- Perplexity `= exp(mean negative log-likelihood per token)`: the effective number of equally likely choices the model is hesitating between. Uniform guessing gives perplexity `V`; the data's true entropy `H` gives a floor of `exp(H)`.
- Multiple choice by log-likelihood: pick the choice with the highest total log-prob (`acc`) or the highest per-token average (`acc_norm`); long answers are penalised by the former.
- Exact match is strict string equality *after* a documented normalisation.
- Bootstrap: resample the test examples with replacement, recompute the metric, and take the 2.5th/97.5th percentiles. For comparing two models evaluate them on the *same* examples and bootstrap the per-example differences (paired), which is much tighter than unpaired.

Everything below runs on a synthetic Markov-chain language so that we know the true answer, with tiny models trained in the notebook (no downloads).

### 1. A synthetic language with a known entropy, and two tiny LMs

**What this cell does (plain language):** we generate text from a sparse random Markov chain over 24 tokens. Because we know the transition matrix we know the *best possible* perplexity (`exp` of the chain's entropy). We then train two small causal transformers - a weak one (16-dim, short training) and a stronger one (64-dim, longer) - plus a **unigram** baseline that ignores context. Three models of clearly different quality give us something to compare.

In [1]:
import torch
torch.set_num_threads(1)  # tiny models: multi-threading overhead only slows them down (and keeps runs deterministic)
import math, copy, numpy as np, pandas as pd, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)
V, L = 24, 24

trans = torch.softmax(torch.randn(V, V) * 2.5, dim=1)          # true next-token distribution given previous token
def gen(n, g=None):
    seq = torch.zeros(n, L, dtype=torch.long); seq[:, 0] = torch.randint(0, V, (n,))
    for t in range(1, L): seq[:, t] = torch.multinomial(trans[seq[:, t - 1]], 1).squeeze(-1)
    return seq
train, test = gen(3000), gen(600)

class TinyLM(nn.Module):
    def __init__(self, d, layers=1, heads=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(L, d)
        self.blocks = nn.ModuleList([nn.ModuleDict(dict(ln1=nn.LayerNorm(d), qkv=nn.Linear(d, 3 * d), proj=nn.Linear(d, d),
                      ln2=nn.LayerNorm(d), fc1=nn.Linear(d, 4 * d), fc2=nn.Linear(4 * d, d))) for _ in range(layers)])
        self.lnf, self.head, self.heads = nn.LayerNorm(d), nn.Linear(d, V), heads
    def forward(self, ids):
        B, T = ids.shape
        h = self.tok(ids) + self.pos(torch.arange(T))
        for b in self.blocks:
            q, k, v = b['qkv'](b['ln1'](h)).chunk(3, -1)
            sp = lambda t: t.view(B, T, self.heads, -1).transpose(1, 2)
            a = F.scaled_dot_product_attention(sp(q), sp(k), sp(v), is_causal=True)
            h = h + b['proj'](a.transpose(1, 2).reshape(B, T, -1))
            h = h + b['fc2'](F.gelu(b['fc1'](b['ln2'](h))))
        return self.head(self.lnf(h))

def train_lm(d, steps, seed):
    torch.manual_seed(seed)
    m = TinyLM(d); o = torch.optim.AdamW(m.parameters(), lr=3e-3)
    for _ in range(steps):
        b = train[torch.randint(0, len(train), (64,))]
        loss = F.cross_entropy(m(b[:, :-1]).reshape(-1, V), b[:, 1:].reshape(-1))
        o.zero_grad(); loss.backward(); o.step()
    return m.eval()

weak, strong = train_lm(16, 40, 1), train_lm(64, 400, 2)
class Unigram(nn.Module):                                      # context-free baseline: next token ~ global frequency
    def __init__(self, data):
        super().__init__()
        f = torch.bincount(data.reshape(-1), minlength=V).float() + 1
        self.register_buffer("logp", (f / f.sum()).log())
    def forward(self, ids): return self.logp.expand(ids.shape[0], ids.shape[1], V)
unigram = Unigram(train).eval()

# the true entropy rate of the chain under its stationary-ish usage (average over observed previous tokens)
prev = test[:, :-1].reshape(-1)
H = -(trans[prev] * trans[prev].log()).sum(1).mean().item()
print(f"true entropy {H:.3f} nats  ->  best achievable perplexity ~ {math.exp(H):.2f} | uniform guessing = {V}")


true entropy 1.437 nats  ->  best achievable perplexity ~ 4.21 | uniform guessing = 24


### 2. Perplexity

**What this cell does (plain language):** perplexity is `exp(mean cross-entropy)` over held-out tokens. We evaluate all models against the uniform guess and the theoretical floor. A good sanity check: no model should ever beat the floor on enough data (if one does, suspect a bug or leakage), and the unigram baseline should land between uniform and the contextual models.

In [2]:
@torch.no_grad()
def token_nll(model, data):
    # per-token negative log-likelihood, shape (N, L-1): predict token t+1 from tokens <= t
    logits = model(data[:, :-1])
    return F.cross_entropy(logits.reshape(-1, V), data[:, 1:].reshape(-1), reduction="none").view(data.shape[0], -1)

models = {"unigram": unigram, "weak LM": weak, "strong LM": strong}
print(f"{'model':<10}{'mean NLL':>10}{'perplexity':>12}")
print(f"{'uniform':<10}{math.log(V):>10.3f}{V:>12.2f}")
for name, m in models.items():
    nll = token_nll(m, test).mean().item()
    print(f"{name:<10}{nll:>10.3f}{math.exp(nll):>12.2f}")
print(f"{'floor':<10}{H:>10.3f}{math.exp(H):>12.2f}")


model       mean NLL  perplexity
uniform        3.178       24.00
unigram        2.918       18.51
weak LM        2.362       10.61
strong LM      1.498        4.47
floor          1.437        4.21


### 3. Perplexity pitfalls: padding in the average, and position dependence

**What this cell does (plain language):** two classic mistakes. (1) Averaging per-sequence means when sequences have different lengths (or including padding tokens) gives a different - wrong - number from the token-weighted mean. We simulate variable lengths with a mask. (2) Early positions have little context and are much harder than later ones, so perplexity depends on the context length you evaluate with; always report it. Here the chain only needs the previous token, so the position effect is small; the weak model is visibly worse at the very first positions.

In [3]:
nll = token_nll(strong, test)                                    # (600, 23)
lengths = torch.randint(4, L, (len(test),))                      # pretend each sequence has its own true length
mask = (torch.arange(L - 1)[None] < (lengths[:, None] - 1)).float()
token_weighted = (nll * mask).sum() / mask.sum()                 # CORRECT: every real token counts once
seq_mean = ((nll * mask).sum(1) / mask.sum(1)).mean()            # per-sequence mean: over-weights short sequences
with_pad = nll.mean()                                            # WRONG: counts padding positions as tokens
print(f"token-weighted ppl {token_weighted.exp().item():.3f} | mean-of-sequence-means ppl {seq_mean.exp().item():.3f} | padding included {with_pad.exp().item():.3f}")

print("\nperplexity by position (strong / weak):")
for lo, hi in [(0, 3), (3, 8), (8, 15), (15, 23)]:
    print(f"  positions {lo:>2}-{hi - 1:>2}: strong {nll[:, lo:hi].mean().exp().item():5.2f} | weak {token_nll(weak, test)[:, lo:hi].mean().exp().item():5.2f}")


token-weighted ppl 4.560 | mean-of-sequence-means ppl 4.555 | padding included 4.471

perplexity by position (strong / weak):
  positions  0- 2: strong  4.48 | weak 11.77
  positions  3- 7: strong  4.73 | weak 10.72
  positions  8-14: strong  4.31 | weak 10.04
  positions 15-22: strong  4.45 | weak 10.64


### 4. Multiple-choice tasks scored by log-likelihood (acc vs. acc_norm)

**What this cell does (plain language):** many benchmarks (HellaSwag, ARC, MMLU in its loglikelihood form) are multiple choice: the model scores each candidate continuation by its log-probability and the argmax is the prediction. We build such a task: a context of 8 tokens, the true continuation (random length 2-6), and three **plausible distractors** (continuations sampled from the chain after other contexts, of random length). Because log-probabilities of longer text are more negative, `acc` (sum) is biased toward short answers; `acc_norm` divides by length. The harness in section 6 uses both.

In [4]:
@torch.no_grad()
def loglik(model, ctx, cont):
    # total log-prob of `cont` given `ctx` (lists of ints)
    ids = torch.tensor([ctx + cont])
    logp = F.log_softmax(model(ids[:, :-1]), dim=-1)[0]
    idx = torch.arange(len(ctx) - 1, len(ctx) + len(cont) - 1)
    return logp[idx, torch.tensor(cont)].sum().item()

def chain_continue(start, n):
    out, cur = [], start
    for _ in range(n):
        cur = torch.multinomial(trans[cur], 1).item(); out.append(cur)
    return out

torch.manual_seed(10)
mc_examples = []
for i in range(300):
    seq = test[i].tolist(); ctx = seq[:8]
    n_true = int(torch.randint(2, 7, (1,)))
    choices = [seq[8:8 + n_true]]
    for _ in range(3):                                           # plausible distractors: chain text from a random other state
        choices.append(chain_continue(int(torch.randint(0, V, (1,))), int(torch.randint(2, 7, (1,)))))
    order = torch.randperm(4).tolist()                           # shuffle so the answer position is random
    mc_examples.append(dict(ctx=ctx, choices=[choices[j] for j in order], label=order.index(0)))
ex0 = mc_examples[0]
print("example context:", ex0["ctx"], "\nchoices:", ex0["choices"], "| correct index:", ex0["label"])
s = [loglik(strong, ex0["ctx"], c) for c in ex0["choices"]]
print("strong LM log-likelihoods:", np.round(s, 2), "-> predicts", int(np.argmax(s)))


example context: [20, 13, 4, 0, 14, 1, 21, 23] 
choices: [[19, 5, 2, 11], [17, 0], [1, 21, 21, 14], [14, 1, 21, 14, 1]] | correct index: 2


strong LM log-likelihoods: [-8.89 -7.25 -5.25 -8.47] -> predicts 2


### 5. Exact match and string normalisation

**What this cell does (plain language):** for generative tasks (QA, math) the usual metric is **exact match (EM)** after normalising both strings; token-level F1 gives partial credit. Normalisation is a policy decision that changes the score, so it must be fixed and documented. Pure Python here: lowercase, strip punctuation and articles, collapse whitespace (the SQuAD recipe).

In [5]:
import re, string, collections
def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, ref): return float(normalize(pred) == normalize(ref))
def f1(pred, ref):
    p, r = normalize(pred).split(), normalize(ref).split()
    common = sum((collections.Counter(p) & collections.Counter(r)).values())
    if common == 0: return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

pairs = [("The Eiffel Tower.", "eiffel tower"), ("Paris, France", "Paris"), ("42", "forty-two"), ("  A cat ", "cat")]
for pred, ref in pairs:
    print(f"pred={pred!r:<22} ref={ref!r:<15} raw== {pred == ref!s:<5} EM {exact_match(pred, ref):.0f}  F1 {f1(pred, ref):.2f}")


pred='The Eiffel Tower.'    ref='eiffel tower'  raw== False EM 1  F1 1.00
pred='Paris, France'        ref='Paris'         raw== False EM 0  F1 0.67
pred='42'                   ref='forty-two'     raw== False EM 0  F1 0.00
pred='  A cat '             ref='cat'           raw== False EM 1  F1 1.00


### 6. A mini evaluation harness

**What this cell does (plain language):** we now put it together in ~40 lines: a `Task` is a list of examples plus a function that scores *one* example with a model and returns one or more metric values; `evaluate` runs every model on every task and keeps **per-example scores** (needed for confidence intervals and paired comparisons later). The model interface is just two methods - `loglikelihood(ctx, cont)` and `generate(ctx, n)` - so any model (including an API) can be plugged in. Tasks: multiple choice (`acc`, `acc_norm`) and a generative task where the model must greedily produce the 3 most likely next tokens, graded by exact match against the chain's own argmax path.

In [6]:
from dataclasses import dataclass
from typing import Callable

class LMAdapter:                                               # the interface a model must implement
    def __init__(self, net): self.net = net
    def loglikelihood(self, ctx, cont): return loglik(self.net, ctx, cont)
    @torch.no_grad()
    def generate(self, ctx, n):
        ids = torch.tensor([ctx])
        for _ in range(n): ids = torch.cat([ids, self.net(ids)[:, -1].argmax(-1, keepdim=True)], dim=1)
        return ids[0, len(ctx):].tolist()

class RandomGuesser:                                           # sanity baseline: scores must be near chance
    def __init__(self, seed=0): self.g = torch.Generator().manual_seed(seed)
    def loglikelihood(self, ctx, cont): return torch.rand(1, generator=self.g).item() * -10
    def generate(self, ctx, n): return torch.randint(0, V, (n,), generator=self.g).tolist()

@dataclass
class Task:
    name: str
    examples: list
    score: Callable            # (model, example) -> dict[str, float]

def score_mc(model, ex):
    ll = np.array([model.loglikelihood(ex["ctx"], c) for c in ex["choices"]])
    ln = np.array([len(c) for c in ex["choices"]])
    return {"acc": float(ll.argmax() == ex["label"]), "acc_norm": float((ll / ln).argmax() == ex["label"])}

gen_examples = []
for i in range(300, 600):
    ctx = test[i, :8].tolist(); path, cur = [], ctx[-1]
    for _ in range(3): cur = trans[cur].argmax().item(); path.append(cur)       # reference = most likely path under the TRUE chain
    gen_examples.append(dict(ctx=ctx, ref=path))
def score_gen(model, ex):
    pred = model.generate(ex["ctx"], 3)
    return {"exact_match": exact_match(" ".join(map(str, pred)), " ".join(map(str, ex["ref"])))}

tasks = [Task("mc_cloze", mc_examples, score_mc), Task("greedy_path", gen_examples, score_gen)]

def evaluate(models, tasks):
    out = {}                                                   # out[model][task][metric] -> np.array over examples
    for mname, model in models.items():
        out[mname] = {}
        for t in tasks:
            rows = [t.score(model, ex) for ex in t.examples]
            out[mname][t.name] = {k: np.array([r[k] for r in rows]) for k in rows[0]}
    return out

harness_models = {"random": RandomGuesser(), "unigram": LMAdapter(unigram), "weak LM": LMAdapter(weak), "strong LM": LMAdapter(strong)}
results = evaluate(harness_models, tasks)
table = pd.DataFrame({m: {f"{t}/{k}": v.mean() for t, d in r.items() for k, v in d.items()} for m, r in results.items()}).round(3)
print(table)
print("\n(chance for 4-way multiple choice = 0.25; greedy_path chance = 1/24^3)")


                         random  unigram  weak LM  strong LM
mc_cloze/acc              0.250    0.280    0.383      0.577
mc_cloze/acc_norm         0.237    0.323    0.450      0.617
greedy_path/exact_match   0.000    0.000    0.580      0.833

(chance for 4-way multiple choice = 0.25; greedy_path chance = 1/24^3)


### 7. Bootstrap confidence intervals for a metric

**What this cell does (plain language):** a single accuracy on 300 examples is an estimate with sampling noise. The **percentile bootstrap** resamples the examples with replacement thousands of times, recomputes the metric, and reports the 2.5th and 97.5th percentiles. We compute 95% CIs for the strong model's metrics and compare against the analytic normal approximation `1.96*sqrt(p(1-p)/n)` - they should agree closely for accuracy-like metrics.

In [7]:
rng = np.random.default_rng(0)
def bootstrap_ci(scores, n_boot=2000, alpha=0.05, rng=rng):
    scores = np.asarray(scores, dtype=float)
    idx = rng.integers(0, len(scores), size=(n_boot, len(scores)))        # n_boot resamples of the example indices
    stats = scores[idx].mean(axis=1)
    return scores.mean(), np.quantile(stats, alpha / 2), np.quantile(stats, 1 - alpha / 2)

for task, metric in [("mc_cloze", "acc"), ("mc_cloze", "acc_norm"), ("greedy_path", "exact_match")]:
    s = results["strong LM"][task][metric]
    m, lo, hi = bootstrap_ci(s)
    half = 1.96 * math.sqrt(m * (1 - m) / len(s))
    print(f"{task}/{metric:<12} {m:.3f}  95% CI [{lo:.3f}, {hi:.3f}]   (normal approx [{m - half:.3f}, {m + half:.3f}])")


mc_cloze/acc          0.577  95% CI [0.520, 0.633]   (normal approx [0.521, 0.633])
mc_cloze/acc_norm     0.617  95% CI [0.563, 0.670]   (normal approx [0.562, 0.672])
greedy_path/exact_match  0.833  95% CI [0.793, 0.873]   (normal approx [0.791, 0.876])


### 8. Is model A really better than model B? Paired vs. unpaired bootstrap

**What this cell does (plain language):** two models are scored on the same examples, so their per-example results are correlated (both find the same questions easy). The **paired** bootstrap resamples examples once and recomputes the *difference* of the two models, so the shared difficulty cancels out. Unpaired treats the two scores as independent samples and yields a wider interval. We compare strong vs weak (a clear gap) and, for contrast, two scorings of the same model (`acc` vs `acc_norm`), where a difference may be within noise.

In [8]:
def paired_bootstrap(a, b, n_boot=5000, rng=np.random.default_rng(1)):
    a, b = np.asarray(a, float), np.asarray(b, float)
    n = len(a); idx = rng.integers(0, n, size=(n_boot, n))
    d = (a[idx] - b[idx]).mean(1)                              # same resampled examples for both models
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())              # two-sided bootstrap p-value (approximate)
    return (a - b).mean(), np.quantile(d, 0.025), np.quantile(d, 0.975), p

def unpaired_ci(a, b, n_boot=5000, rng=np.random.default_rng(2)):
    a, b = np.asarray(a, float), np.asarray(b, float)
    da = a[rng.integers(0, len(a), (n_boot, len(a)))].mean(1); db = b[rng.integers(0, len(b), (n_boot, len(b)))].mean(1)
    d = da - db
    return np.quantile(d, 0.025), np.quantile(d, 0.975)

cmp = [("mc acc: strong - weak", results["strong LM"]["mc_cloze"]["acc"], results["weak LM"]["mc_cloze"]["acc"]),
       ("greedy EM: strong - weak", results["strong LM"]["greedy_path"]["exact_match"], results["weak LM"]["greedy_path"]["exact_match"]),
       ("strong: acc_norm - acc", results["strong LM"]["mc_cloze"]["acc_norm"], results["strong LM"]["mc_cloze"]["acc"])]
for name, a, b in cmp:
    d, lo, hi, p = paired_bootstrap(a, b)
    ulo, uhi = unpaired_ci(a, b)
    print(f"{name:<26} diff {d:+.3f} | paired 95% CI [{lo:+.3f}, {hi:+.3f}] p~{p:.3f} | unpaired CI [{ulo:+.3f}, {uhi:+.3f}]")


mc acc: strong - weak      diff +0.193 | paired 95% CI [+0.140, +0.250] p~0.000 | unpaired CI [+0.113, +0.270]
greedy EM: strong - weak   diff +0.253 | paired 95% CI [+0.193, +0.317] p~0.000 | unpaired CI [+0.183, +0.323]
strong: acc_norm - acc     diff +0.040 | paired 95% CI [-0.013, +0.093] p~0.150 | unpaired CI [-0.037, +0.117]


### 9. Bootstrapping perplexity: resample sequences, not tokens

**What this cell does (plain language):** tokens within a sequence are correlated, so the right unit to resample is the *sequence* (a cluster bootstrap). We compute the per-sequence mean NLL, resample sequences, and take `exp` of each resampled mean to get a CI on perplexity. Resampling individual tokens would make the interval look artificially narrow.

In [9]:
def ppl_ci(model, data, n_boot=2000, seed=0, by="sequence"):
    nll = token_nll(model, data)                                # (N, L-1)
    g = torch.Generator().manual_seed(seed)
    if by == "sequence":
        per_seq = nll.mean(1)
        idx = torch.randint(0, len(per_seq), (n_boot, len(per_seq)), generator=g)
        stats = per_seq[idx].mean(1).exp()
    else:                                                       # token-level resample (too optimistic)
        flat = nll.reshape(-1)
        idx = torch.randint(0, len(flat), (n_boot, len(flat)), generator=g)
        stats = flat[idx].mean(1).exp()
    return nll.mean().exp().item(), stats.quantile(0.025).item(), stats.quantile(0.975).item()

for by in ("sequence", "token"):
    for name, m in [("weak LM", weak), ("strong LM", strong)]:
        p_, lo, hi = ppl_ci(m, test, by=by)
        print(f"resample by {by:<8} {name:<10} ppl {p_:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  width {hi - lo:.3f}")


resample by sequence weak LM    ppl 10.607  95% CI [10.361, 10.858]  width 0.497
resample by sequence strong LM  ppl 4.471  95% CI [4.363, 4.573]  width 0.211


resample by token    weak LM    ppl 10.607  95% CI [10.422, 10.799]  width 0.377


resample by token    strong LM  ppl 4.471  95% CI [4.369, 4.579]  width 0.210


### 10. How many examples do you need? CI width vs. test-set size

**What this cell does (plain language):** the CI half-width of an accuracy shrinks like `1/sqrt(n)`: quadrupling the test set only halves the uncertainty. We bootstrap sub-samples of different sizes (drawn from a simulated model with 70% accuracy) and compare to the formula. The practical consequence: with a couple of hundred examples you cannot distinguish models that are 2-3 points apart, no matter how carefully you run the evaluation.

In [10]:
rng2 = np.random.default_rng(3)
pop = (rng2.random(100000) < 0.70).astype(float)               # per-example correctness of a model with 70% true accuracy
print(f"{'n':>6} {'bootstrap half-width':>21} {'1.96*sqrt(p(1-p)/n)':>21}")
for n in (50, 200, 800, 3200, 12800):
    s = rng2.choice(pop, n, replace=False)
    _, lo, hi = bootstrap_ci(s, n_boot=1000, rng=rng2)
    print(f"{n:>6} {(hi - lo) / 2:>21.4f} {1.96 * math.sqrt(0.7 * 0.3 / n):>21.4f}")
need = (1.96 * math.sqrt(0.7 * 0.3) / 0.01) ** 2
print(f"examples needed for a +/-1 point half-width at 70% accuracy: ~{need:,.0f}")


     n  bootstrap half-width   1.96*sqrt(p(1-p)/n)
    50                0.1300                0.1270
   200                0.0650                0.0635
   800                0.0319                0.0318
  3200                0.0163                0.0159


 12800                0.0077                0.0079
examples needed for a +/-1 point half-width at 70% accuracy: ~8,067


### 11. Contamination check: n-gram overlap between train and test

**What this cell does (plain language):** if test examples (or near copies) appear in the training data, scores are inflated and meaningless. A standard first check is n-gram overlap: what fraction of each test item's n-grams occur in the training set? We build a clean test set and a 'leaked' one in which 10% of items are copied from the training data, and show that the overlap statistic flags the leak. We use 20-grams (items are 24 tokens long) so that chance overlaps in this synthetic language are essentially zero; real contamination checks use 8-13-grams on natural text, which is much more diverse.

In [11]:
def ngrams(seq, n=20): return {tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)}
train_grams = set()
for s in train.tolist(): train_grams |= ngrams(s)

def overlap(data):
    return np.array([len(ngrams(s) & train_grams) / len(ngrams(s)) for s in data.tolist()])

leaked = test.clone()
leak_idx = torch.randperm(len(test))[:60]
leaked[leak_idx] = train[torch.randperm(len(train))[:60]]       # 10% of test items copied from train
ov_clean, ov_leaked = overlap(test), overlap(leaked)
print(f"clean test : items with any 20-gram overlap {np.mean(ov_clean > 0):.3f}")
print(f"leaked test: items with any 20-gram overlap {np.mean(ov_leaked > 0):.3f}  (expected ~0.10)")
print("flagged items are exactly the leaked ones:", set(np.where(ov_leaked > 0)[0]) >= set(leak_idx.tolist()))


clean test : items with any 20-gram overlap 0.000
leaked test: items with any 20-gram overlap 0.100  (expected ~0.10)
flagged items are exactly the leaked ones: True


## Common bugs

- **Averaging over padding tokens**
  Padded positions contribute fake zero-loss tokens (or garbage) and distort perplexity. Fix: pass a loss mask / `ignore_index`, and compute the token-weighted mean (section 3 shows the gap).

- **Comparing perplexities across tokenizers**
  Perplexity is per *token*; a model with a bigger vocabulary has fewer tokens per text and a larger per-token perplexity for the same quality. Fix: compare only with the same tokenizer, or convert to bits-per-character/byte.

- **Averaging per-sequence perplexities**
  The mean of per-sequence perplexities is not the corpus perplexity (exp is convex). Fix: average NLL over tokens, then exponentiate once.

- **Using `acc` when answer lengths differ**
  Sum-log-likelihood penalises long answers, so scores depend on the answer-length distribution. Fix: report `acc_norm` too, and say which normalisation (per-token or per-character) you use.

- **Inconsistent normalisation for exact match**
  Changing lowercase/punctuation/article handling between runs changes EM by several points. Fix: freeze one normaliser (as in `normalize`) and use it for all models.

- **Reporting a single number with no interval**
  On a few hundred examples, differences of 2-3 points are noise (section 10). Fix: report bootstrap CIs, and use a paired test when comparing models on the same examples.

- **Test-set contamination or tuning on the test set**
  Overlap with training data, or choosing prompts/hyper-parameters to maximise the test score, inflates results. Fix: n-gram overlap checks, a held-out set that is touched once, and a separate dev set for choices.

- **Forgetting `model.eval()` / `torch.no_grad()` during evaluation**
  Dropout stays active (noisy scores) and memory balloons from stored graphs. Fix: `model.eval()` and `@torch.no_grad()` (or `inference_mode`) in every eval path.

- **Not including a baseline**
  A 55% accuracy means little without chance and trivial baselines (random, unigram, majority class) next to it. Fix: always evaluate baselines through the same harness, as in section 6.

## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1.** Compute the bits-per-token of the strong model on the test set (`NLL / ln 2`) and relate it to perplexity (`2 ** bits`).

In [12]:
# Your attempt for Exercise 1 here


**Solution 1**

In [13]:
# Solution 1
nll_nats = token_nll(strong, test).mean().item()
bits = nll_nats / math.log(2)
print("bits/token:", round(bits, 3), "| 2**bits:", round(2 ** bits, 3), "| perplexity:", round(math.exp(nll_nats), 3))


bits/token: 2.161 | 2**bits: 4.471 | perplexity: 4.471


**Exercise 2.** Add an `acc_top2` metric to the multiple-choice scorer (correct if the label is among the 2 highest-scoring choices) and evaluate the strong model with a 95% bootstrap CI.

In [14]:
# Your attempt for Exercise 2 here


**Solution 2**

In [15]:
# Solution 2
def score_mc2(model, ex):
    ll = np.array([model.loglikelihood(ex["ctx"], c) for c in ex["choices"]])
    return {"acc_top2": float(ex["label"] in np.argsort(-ll)[:2])}
vals = np.array([score_mc2(harness_models["strong LM"], e)["acc_top2"] for e in mc_examples])
print("acc_top2 mean, 95% CI:", [round(float(v), 3) for v in bootstrap_ci(vals)])


acc_top2 mean, 95% CI: [0.807, 0.763, 0.85]


**Exercise 3.** Show that the unpaired comparison is wider than the paired one for strong vs weak on `mc_cloze/acc_norm`, and explain in a comment which examples drive the correlation.

In [16]:
# Your attempt for Exercise 3 here


**Solution 3**

In [17]:
# Solution 3
a, b = results["strong LM"]["mc_cloze"]["acc_norm"], results["weak LM"]["mc_cloze"]["acc_norm"]
d, lo, hi, p = paired_bootstrap(a, b); ulo, uhi = unpaired_ci(a, b)
print(f"paired width {hi - lo:.3f} vs unpaired width {uhi - ulo:.3f}")
print("correlation of per-example correctness between the models:", round(float(np.corrcoef(a, b)[0, 1]), 3))
# Positive correlation => easy examples are easy for both, so the difference has lower variance than two independent scores.


paired width 0.113 vs unpaired width 0.157
correlation of per-example correctness between the models: 0.479


**Exercise 4.** Implement a *stratified* report: accuracy of the strong model on `mc_cloze` split by correct-answer length (2..6 tokens). Does `acc` depend on length more than `acc_norm`?

In [18]:
# Your attempt for Exercise 4 here


**Solution 4**

In [19]:
# Solution 4
lens = np.array([len(e["choices"][e["label"]]) for e in mc_examples])
acc, accn = results["strong LM"]["mc_cloze"]["acc"], results["strong LM"]["mc_cloze"]["acc_norm"]
for n in range(2, 7):
    sel = lens == n
    print(f"correct answer length {n}: n={sel.sum():3d} acc {acc[sel].mean():.2f} acc_norm {accn[sel].mean():.2f}")


correct answer length 2: n= 63 acc 0.84 acc_norm 0.59
correct answer length 3: n= 49 acc 0.78 acc_norm 0.69
correct answer length 4: n= 62 acc 0.60 acc_norm 0.66
correct answer length 5: n= 58 acc 0.41 acc_norm 0.59
correct answer length 6: n= 68 acc 0.31 acc_norm 0.57


**Exercise 5.** Evaluate a *calibration* quantity: the strong model's mean predicted probability of the true next token vs. the empirical frequency that its argmax is right (a one-bin Expected Calibration Error).

In [20]:
# Your attempt for Exercise 5 here


**Solution 5**

In [21]:
# Solution 5
with torch.no_grad():
    probs = F.softmax(strong(test[:, :-1]), dim=-1)
    conf, pred = probs.max(-1)
    correct = (pred == test[:, 1:]).float()
print("mean confidence:", round(conf.mean().item(), 3), "| accuracy:", round(correct.mean().item(), 3), "| |gap| (1-bin ECE):", round(abs(conf.mean().item() - correct.mean().item()), 3))


mean confidence: 0.565 | accuracy: 0.547 | |gap| (1-bin ECE): 0.018


**Exercise 6.** Add a `Task` that measures perplexity per example (negative mean NLL, so higher is better) and use the harness to compare all four models, including a bootstrap CI for the strong model.

In [22]:
# Your attempt for Exercise 6 here


**Solution 6**

In [23]:
# Solution 6
ppl_examples = [dict(seq=test[i].tolist()) for i in range(200)]
def score_nll(model, ex):
    s = ex["seq"]
    return {"neg_nll": float(np.mean([model.loglikelihood(s[:t], [s[t]]) for t in range(1, len(s))]))}
res_p = evaluate({k: v for k, v in harness_models.items() if k != "random"}, [Task("lm", ppl_examples, score_nll)])
for m, r in res_p.items(): print(f"{m:<10} mean neg-NLL {r['lm']['neg_nll'].mean():.3f}")
print("strong LM CI:", [round(float(v), 3) for v in bootstrap_ci(res_p["strong LM"]["lm"]["neg_nll"])])


unigram    mean neg-NLL -2.914
weak LM    mean neg-NLL -2.360
strong LM  mean neg-NLL -1.504
strong LM CI: [-1.504, -1.545, -1.461]
